# Galaxy-X-os  —  One-Click Colab Pipeline (SCALE x ODYSSEY)

Classify raw astronomical images into **5 celestial categories** with EfficientNet-B3.

**How to run:** `Runtime → Change runtime type → GPU (T4)`, then `Runtime → Run all`.

Pipeline: clone → install → (optional Kaggle) → prepare data → train → evaluate → Grad-CAM → download results.

Every cell is idempotent — re-running is safe.

## Cell 1 — Clone repo + install dependencies

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Srujan0798/Galaxy-X-os"
REPO_DIR = "/content/Galaxy-X-os"

# Detect if we are already inside the repo (e.g. mounted Drive); else clone.
if os.path.exists("src/prepare_data.py"):
    REPO_DIR = os.getcwd()
    print(f"Already inside repo at {REPO_DIR}")
elif os.path.exists(os.path.join(REPO_DIR, "src/prepare_data.py")):
    print(f"Repo already cloned at {REPO_DIR}")
else:
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

# Core deps + extras needed for the real-first data pipeline.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astroNN", "kagglehub", "h5py"], check=False)
print("Dependencies installed.")

## Cell 1b — Verify GPU is present (fail loudly if not)

Training EfficientNet-B3 on CPU is impractically slow. If this fails, set
`Runtime → Change runtime type → Hardware accelerator → GPU` and re-run.

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "\n\n  NO GPU DETECTED.\n"
    "  Fix: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4),\n"
    "  then Runtime -> Run all.\n"
)
print("GPU OK:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)

## Cell 2 — (Optional) Upload `kaggle.json` for REAL nebula / cluster / planetary

Spiral & elliptical galaxies always come from the **real Galaxy10** dataset (no key needed).

For **real** nebula / star_cluster / planetary images you may upload a Kaggle API token
(`kaggle.json` from https://www.kaggle.com/settings → *Create New Token*).

**If you skip this**, those three classes use a clearly-labelled **procedural fallback** so the
pipeline never breaks. The choice is recorded honestly in `data/processed/DATA_MANIFEST.json`.

In [ ]:
import os, json
from pathlib import Path

USE_KAGGLE = False  # set True to be prompted for an upload

if USE_KAGGLE:
    try:
        from google.colab import files
        print("Select your kaggle.json ...")
        up = files.upload()
        if "kaggle.json" in up:
            kdir = Path.home() / ".kaggle"
            kdir.mkdir(exist_ok=True)
            (kdir / "kaggle.json").write_bytes(up["kaggle.json"])
            os.chmod(kdir / "kaggle.json", 0o600)
            print("kaggle.json installed -> REAL Kaggle download will be attempted.")
    except Exception as e:
        print(f"Kaggle upload skipped ({e}).")

has_creds = (Path.home() / ".kaggle" / "kaggle.json").exists() or bool(
    os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
print("Kaggle credentials present:", has_creds,
      "->", "REAL nebula/cluster/planetary attempted" if has_creds
      else "PROCEDURAL fallback for nebula/cluster/planetary")

## Cell 3 — Prepare data (real-first, safe-fallback, idempotent)

Builds `data/processed/{train,val,test}/<class>/`, runs a disjoint 80/10/10
stratified split with an MD5 leakage check, and writes `DATA_MANIFEST.json`
+ `class_weights.json`.

In [ ]:
!python src/prepare_data.py --per-class 500

import json
with open("data/processed/DATA_MANIFEST.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Cell 4 — Train (full EfficientNet-B3 fine-tune on the GPU)

Writes `checkpoints/best_model.pth` (best val accuracy). Target: **> 80%** val accuracy.
Uses `configs/config.yaml` (progressive unfreezing, OneCycleLR, mixed precision, early stopping).

In [ ]:
!python src/train.py

import torch
ckpt = torch.load("checkpoints/best_model.pth", map_location="cpu", weights_only=True)
print(f"\nBest val accuracy: {ckpt['best_val_acc']:.4f}  (epoch {ckpt['epoch'] + 1})")

## Cell 5 — Evaluate (standard + Test-Time Augmentation)

Shows `results/evaluation_results.json` and the confusion matrix inline.

In [ ]:
!python src/evaluate.py

import json
from IPython.display import Image as IPyImage, display
with open("results/evaluation_results.json") as f:
    print(json.dumps(json.load(f), indent=2))
for p in ["results/confusion_matrix.png", "results/per_class_metrics.png"]:
    try:
        display(IPyImage(filename=p))
    except Exception as e:
        print(f"(could not display {p}: {e})")

## Cell 6 — Grad-CAM from the REAL trained model

In [ ]:
!python src/gradcam.py

import glob
from IPython.display import Image as IPyImage, display
cams = sorted(glob.glob("results/gradcam/*.png"))
print(f"Generated {len(cams)} Grad-CAM images.")
for p in cams[:4]:
    display(IPyImage(filename=p))

## Cell 7 — Zip results + checkpoint and download

After download: **unzip `results.zip` into the repo root, then commit.**

In [ ]:
import zipfile, os

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk("results"):
        for fn in fnames:
            fp = os.path.join(root, fn)
            z.write(fp, fp)
    if os.path.exists("checkpoints/best_model.pth"):
        z.write("checkpoints/best_model.pth", "checkpoints/best_model.pth")

print("Wrote results.zip:", round(os.path.getsize("results.zip") / 1e6, 1), "MB")
print("\nAfter download:\n  1. unzip results.zip into the repo root\n"
      "  2. git add results checkpoints && git commit -m 'Add trained model + results'")

try:
    from google.colab import files
    files.download("results.zip")
except Exception as e:
    print(f"(auto-download unavailable: {e} — download results.zip from the Files panel)")